In [1]:
from pydantic import BaseModel, Field
from typing import List
from openai import OpenAI
import pandas as pd
from dotenv import load_dotenv
import os
import time
from langchain_core.prompts import ChatPromptTemplate
load_dotenv()

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url=os.getenv("MODEL_BASE_URL")
)
DATA_PATH=os.getenv("DATA_PATH")

class VehicleIssue(BaseModel):
    issue_id: str = Field(description="Unique identifier, e.g. 'p0420-001'")
    issue_name: str = Field(description="Name of the issue, e.g. 'Catalyst System Efficiency Below Threshold'")
    obd_code: str = Field(description="Real OBD-II code, e.g. 'P0420', 'P0300'. Use 'N/A' if not code-based")
    system: str = Field(description="Engine, Transmission, Brakes, Electrical, Suspension, Cooling, etc.")
    component: str = Field(description="Specific part, e.g. 'Catalytic Converter', 'Brake Pads', 'Alternator'")
    severity: str = Field(description="Low, Medium, High, Critical")
    symptoms: str = Field(description="Comma-separated list, e.g. 'Check engine light, rough idle, reduced power'")
    likely_causes: str = Field(description="Comma-separated list, e.g. 'Faulty O2 sensor, worn spark plugs'")
    diagnostic_steps: str = Field(description="Step-by-step instructions to confirm the diagnosis")
    diy_or_mechanic: str = Field(description="DIY, Mechanic Recommended, Mechanic Required")

class VehicleIssueDataset(BaseModel):
    issues: List[VehicleIssue]

In [2]:
all_issues = []
batch_size = 10
num_batches = 5

def generate_batch(batch_prompt, max_retries=3):
    for attempt in range(max_retries):
        try:
            response = openai_client.chat.completions.parse(
                model=os.getenv("AI_MODEL"),
                messages=[{"role": "user", "content": batch_prompt}],
                response_format=VehicleIssueDataset,
                reasoning_effort="low",
            )
            return response.choices[0].message.parsed.issues
        except Exception as e:
            print(f"  Attempt {attempt+1} failed: {e}")
            time.sleep(1)
    print("  All retries failed, skipping batch.")
    return []

all_issues = []
batch_size = 10
num_batches = 5

for i in range(num_batches):
    batch_prompt = f"""
Generate exactly {batch_size} diverse, realistic vehicle issues (batch {i+1} of {num_batches}).
Cover different systems (engine, transmission, brakes, electrical, suspension, cooling).
Use real OBD-II codes where applicable (e.g. P0171, P0300, P0420, P0455), 'N/A' for non-code issues.
Vary severity levels and mix DIY-fixable issues with ones that require a mechanic.
Output valid JSON only — field names must exactly match the schema, with no markdown formatting (no asterisks, no bold) in keys or values.
Avoid generating an issue with the same issue_name and obd_code as any previously generated issue.
Previously generated issues:
{[(iss.issue_name, iss.obd_code) for iss in all_issues]}
""".strip()

    batch_issues = generate_batch(batch_prompt)
    print(f"Batch {i+1}: requested {batch_size}, got {len(batch_issues)}")
    all_issues.extend(batch_issues)

df = pd.DataFrame([issue.model_dump() for issue in all_issues])

# Remove duplicate issues
df = df.drop_duplicates(
    subset=["issue_name", "obd_code"],
    keep="first"
)

# Give unique IDs AFTER removing duplicates
df["issue_id"] = [
    f"issue_{i:04d}"
    for i in range(1, len(df) + 1)
]

df.to_csv(DATA_PATH, index=False)

print(f"Generated {len(df)} unique vehicle issues")
print(f"Unique IDs: {df['issue_id'].nunique()}")

Batch 1: requested 10, got 10
Batch 2: requested 10, got 10
Batch 3: requested 10, got 10
  Attempt 1 failed: Error code: 400 - {'error': {'message': "Failed to validate JSON. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': ''}}
Batch 4: requested 10, got 10
Batch 5: requested 10, got 10
Generated 50 unique vehicle issues
Unique IDs: 50


In [3]:
from openai import OpenAI

def llm(prompt, model=os.getenv("AI_MODEL")):
    response = openai_client.responses.create(
        model=model,
        input=[{"role": "user", "content": prompt}]
    )

    return response.output_text

In [2]:
from versions.v2.rag import query,evaluate_relevance

question = "who is john"
answer = query(question)
print(answer)

OperationalError: could not translate host name "postgres" to address: Name or service not known


In [5]:
prompt1_template = """
You are a vehicle diagnostic expert generating evaluation questions.
For the issue below, generate 2 questions that a user might ask.
Return the result as a JSON array with objects containing 'issue_id' and 'question' fields.
Issue: {issue_name}
OBD Code: {obd_code}
System: {system}
Component: {component}
Symptoms: {symptoms}
Likely causes: {likely_causes}
Diagnostic steps: {diagnostic_steps}
""".strip()
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url=os.getenv("MODEL_BASE_URL")
)

from tqdm.auto import tqdm
import json

df = pd.read_csv(DATA_PATH)

results = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    prompt = prompt1_template.format(**row.to_dict())
    response = openai_client.chat.completions.create(
        model=os.getenv("AI_MODEL"),
        messages=[{"role": "user", "content": prompt}]
    )
    print(repr(response.choices[0].message.content))
    content = response.choices[0].message.content.strip()

    # strip markdown code fences if the model added them
    if content.startswith("```"):
        content = content.strip("`")
        if content.startswith("json"):
            content = content[4:].strip()

    try:
        questions = json.loads(content)
    except json.JSONDecodeError:
        print(f"Failed to parse issue_id={row['issue_id']}: {content[:200]}")
        continue

    for q in questions:
        q["issue_id"] = row["issue_id"]
        results.append(q)

df_questions = pd.DataFrame(results)
df_questions.to_csv("data/ground-truth-retrieval.csv", index=False)

  0%|          | 0/50 [00:00<?, ?it/s]

'```json\n[\n  {\n    "issue_id": "P0171",\n    "question": "How can I determine if a vacuum leak is the cause of the P0171 (System Too Lean Bank 1) code?"\n  },\n  {\n    "issue_id": "P0171",\n    "question": "What voltage range should I see on the MAF sensor when the engine is idling, and how do I test it for a P0171 condition?"\n  }\n]\n```'
'[\n  {\n    "issue_id": "P0300",\n    "question": "How can I tell if my spark plugs are the cause of the random/multiple cylinder misfire (P0300) and when should they be replaced?"\n  },\n  {\n    "issue_id": "P0300",\n    "question": "What is the correct fuel pressure for my engine, and how do I test it to rule out low fuel pressure as a cause of the P0300 code?"\n  }\n]'
'[\n  {\n    "issue_id": "P0420",\n    "question": "Why is the check engine light on and my vehicle experiencing reduced performance with a P0420 code?"\n  },\n  {\n    "issue_id": "P0420",\n    "question": "What diagnostic steps can I take to determine if my catalytic conver

In [6]:
import pandas as pd
from tqdm.auto import tqdm

df_question = pd.read_csv("data/ground-truth-retrieval.csv")
ground_truth = df_question.to_dict(orient="records")

In [7]:
def hit_rate(relevance_total):
    cnt = 0
    for line in relevance_total:
        if True in line:
            cnt += 1
    return cnt / len(relevance_total)

def mrr(relevance_total):
    total_score = 0.0
    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank]:
                total_score += 1 / (rank + 1)
                break
    return total_score / len(relevance_total)

def evaluate(ground_truth, search_function):
    relevance_total = []
    for q in tqdm(ground_truth):
        doc_id = q["issue_id"]
        results = search_function(q)
        relevance = [d["issue_id"] == doc_id for d in results]
        relevance_total.append(relevance)
    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total)
    }

In [8]:
def sqllite_search(query, boost=None):
    if boost is None:
        boost = {}

    results = text_index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=5
    )
    return results

In [9]:
from project.ingest_fresh_or_load_data import load_or_build_text_index,create_or_load_vectorstore
text_index=load_or_build_text_index()
evaluate(ground_truth, lambda q: sqllite_search(q["question"],boost={'issue_name': 2.762878639196613, 'obd_code': 1.8952365779659295, 'system': 1.2564655180690536, 'component': 1.2285176926271535, 'severity': 2.0134891542200783, 'symptoms': 2.53495307231972, 'likely_causes': 1.9505921600615506, 'diagnostic_steps': 0.07927706902520026, 'diy_or_mechanic': 2.2724329092151416}))


Found existing index at ./issues.db, loading it.


  0%|          | 0/100 [00:00<?, ?it/s]

{'hit_rate': 0.97, 'mrr': 0.8728333333333333}

In [10]:
df_validation = df_question[:50]
df_test = df_question[50:]

gt_val = df_validation.to_dict(orient="records")
gt_test = df_test.to_dict(orient="records")

In [11]:
print(len(df_test))

50


In [12]:
import random

def simple_optimize(param_ranges, objective_function, n_iterations=20):
    best_params = None
    best_score = float("-inf")

    for _ in range(n_iterations):
        current_params = {}
        for field, (low, high) in param_ranges.items():
            current_params[field] = random.uniform(low, high)

        current_score = objective_function(current_params)

        if current_score > best_score:
            best_score = current_score
            best_params = current_params

    return best_params

param_ranges = {
    "issue_name": (0.0, 3.0),
    "obd_code": (0.0, 3.0),
    "system": (0.0, 3.0),
    "component": (0.0, 3.0),
    "severity": (0.0, 3.0),
    "symptoms": (0.0, 3.0),
    "likely_causes": (0.0, 3.0),
    "diagnostic_steps": (0.0, 3.0),
    "diy_or_mechanic": (0.0, 3.0),
}
def objective(boost_params):
    def search_function(q):
        return sqllite_search(q["question"], boost=boost_params)

    results = evaluate(gt_val, search_function)
    return results["hit_rate"]

best_params = simple_optimize(param_ranges, objective, n_iterations=20)
print("Best boost parameters:", best_params)

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

...

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

Best boost parameters: {'issue_name': 0.7910349460596868, 'obd_code': 1.923332285615703, 'system': 2.4032675292713974, 'component': 1.6592385502750262, 'severity': 0.2576847806355982, 'symptoms': 1.0877929345295587, 'likely_causes': 2.7370957842114443, 'diagnostic_steps': 1.2170080334390445, 'diy_or_mechanic': 0.2815099926046304}


In [19]:


def search(query):
    boost = {'issue_name': 0.7910349460596868, 'obd_code': 1.923332285615703, 'system': 2.4032675292713974, 'component': 1.6592385502750262, 'severity': 0.2576847806355982, 'symptoms': 1.0877929345295587, 'likely_causes': 2.7370957842114443, 'diagnostic_steps': 1.2170080334390445, 'diy_or_mechanic': 0.2815099926046304}


    results = text_index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=5
    )

    return results

In [20]:
evaluate(gt_test, lambda q: search(q["question"]))


  0%|          | 0/50 [00:00<?, ?it/s]

{'hit_rate': 1.0, 'mrr': 0.9116666666666667}

In [15]:
prompt2_template = """
You are an expert evaluator for a RAG system.
Your task is to analyze the relevance of the generated answer to the given question.
Based on the relevance of the generated answer, you will classify it
as 'NON_RELEVANT', 'PARTLY_RELEVANT', or 'RELEVANT'.

Here is the data for evaluation:

Question: {question}
Generated Answer: {answer_llm}

Please analyze the content and context of the generated answer in relation to the question
and provide your evaluation in parsable JSON without using code blocks:

{{
  "Relevance": "NON_RELEVANT" | "PARTLY_RELEVANT" | "RELEVANT",
  "Explanation": "[Provide a brief explanation for your evaluation]"
}}
""".strip()

In [16]:
import json
from tqdm.auto import tqdm

df_sample = df_question.sample(n=3, random_state=1)
sample = df_sample.to_dict(orient="records")

evaluations = []

for record in tqdm(sample):
    question = record["question"]
    answer_llm = query(question)

    json_eval, tokens = evaluate_relevance(question, answer_llm)

    evaluation = json_eval

    evaluations.append((record, answer_llm, evaluation))

  0%|          | 0/3 [00:00<?, ?it/s]

text_search: count 3 [{'issue_id': 'issue_0041', 'content': 'Issue: Mass Air Flow Sensor Circuit Intermittent\nSymptoms: Rough idle, reduced power, stalling\nLikely Causes: Dirty MAF sensor, cracked wiring harness, faulty sensor\nDiagnostic Steps: 1. Scan for P0101 code. 2. Inspect MAF sensor connector for corrosion or loose pins. 3. Measure voltage signal while engine runs (should fluctuate 0.5-4.5V). 4. Clean MAF sensor with MAF cleaner. 5. Replace sensor if voltage out of range.', 'source': 'text_search'}, {'issue_id': 'issue_0035', 'content': 'Issue: Hybrid Battery Pack Voltage Imbalance\nSymptoms: Reduced electric assist, warning lights, limited driving range\nLikely Causes: Cell imbalance, high voltage fault, BMS failure\nDiagnostic Steps: 1. Use manufacturer-specific scan tool to read battery module voltages.\n2. Identify modules outside tolerance.\n3. Perform balance procedure if supported.\n4. Replace faulty module or BMS if imbalance persists.\n5. Clear codes and verify perfo

In [17]:
df_eval = pd.DataFrame(evaluations, columns=["record", "answer", "evaluation"])

df_eval["issue_id"] = df_eval.record.apply(lambda d: d["issue_id"])
df_eval["question"] = df_eval.record.apply(lambda d: d["question"])
df_eval["relevance"] = df_eval.evaluation.apply(lambda d: d["Relevance"])
df_eval["explanation"] = df_eval.evaluation.apply(lambda d: d["Explanation"])

df_eval.relevance.value_counts(normalize=True)

relevance
RELEVANT    1.0
Name: proportion, dtype: float64

In [18]:

import os
model_name = os.getenv("AI_MODEL")
out_path = f"data/rag-eval-groq/{model_name}.csv"
os.makedirs(os.path.dirname(out_path), exist_ok=True)
df_eval.to_csv(out_path, index=False)

